# A Minimal TypeScript Frontend

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/10-minimal-typescript-frontend.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic fastapi httpx pydantic uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 10 - A Minimal TypeScript Frontend
Phase 06: Shipping

FastAPI backend that serves the two endpoints the TypeScript client expects:
  POST /generate  - sync generation, returns JSON {text: str}
  GET  /stream    - SSE streaming, yields data: <token> lines

Also mounts the frontend/ directory as static files at /ui.

Run:
    uv pip install fastapi uvicorn anthropic
    ANTHROPIC_API_KEY=sk-... uvicorn main:app --reload

Then open http://localhost:8000/ui/ in your browser.
Or serve the frontend separately:
    python -m http.server 3000 --directory frontend/
"""
import os

import anthropic
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel

app = FastAPI(title="AI Frontend Backend", version="1.0.0")

# CORS: required when frontend is on a different port (e.g., localhost:3000)
# In production, replace "*" with your actual frontend domain.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL = "claude-3-5-haiku-20241022"

### Data models

In [ ]:
class GenerateRequest(BaseModel):
    prompt: str


class GenerateResponse(BaseModel):
    text: str

### Sync endpoint (called via fetch)

In [ ]:
@app.post("/generate", response_model=GenerateResponse)
async def generate(request: GenerateRequest) -> GenerateResponse:
    """
    Synchronous generation. Returns the full response as JSON.
    The TypeScript client calls this with fetch() and waits for the response.
    """
    message = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": request.prompt}],
    )
    return GenerateResponse(text=message.content[0].text)

### Streaming endpoint (called via EventSource)

In [ ]:
@app.get("/stream")
async def stream(prompt: str) -> StreamingResponse:
    """
    Streaming generation via Server-Sent Events.
    The TypeScript client uses EventSource (GET only).
    Sends: data: <token>\\n\\n for each token.
    Sends: data: [DONE]\\n\\n when the stream ends.

    Note: prompt passed as query parameter because EventSource is GET-only.
    """

    def generate_stream():
        with client.messages.stream(
            model=MODEL,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
        ) as stream_ctx:
            for text in stream_ctx.text_stream:
                # SSE format: "data: <content>\n\n"
                # Replace newlines in content to avoid breaking SSE framing
                safe_text = text.replace("\n", " ")
                yield f"data: {safe_text}\n\n"
        # Signal to the client that the stream is complete
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        generate_stream(),
        media_type="text/event-stream",
        headers={
            # Prevent buffering in nginx/proxies
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
        },
    )

### Health check

In [ ]:
@app.get("/health")
async def health() -> dict[str, str]:
    return {"status": "ok"}

### Static files (mount last so API routes take precedence)

In [ ]:
# Uncomment when you have a frontend/ directory next to main.py:
# app.mount("/ui", StaticFiles(directory="frontend", html=True), name="ui")


# ---------------------------------------------------------------------------
# Quick test without a browser
# ---------------------------------------------------------------------------

### Demo

In [ ]:
import httpx

BASE = "http://localhost:8000"

print("=== Test: sync /generate ===")
resp = httpx.post(f"{BASE}/generate", json={"prompt": "Say hello in one sentence."})
print(f"Status: {resp.status_code}")
print(f"Response: {resp.json()['text']}")

print("\n=== Test: streaming /stream ===")
print("Tokens: ", end="", flush=True)
with httpx.stream("GET", f"{BASE}/stream", params={"prompt": "Count from 1 to 5."}) as r:
    for line in r.iter_lines():
        if line.startswith("data: "):
            token = line[6:]
            if token == "[DONE]":
                break
            print(token, end="", flush=True)
print()

---

## Self-Check

Answer these without running code first:

**1. What is the root cause and the correct fix?**

- A. The server is buffering the response. Add 'X-Accel-Buffering: no' to the response headers.
- B. fetch() waits for the full response body before resolving. Switch to EventSource for SSE endpoints.
- C. The DOM update is blocked by a CSS animation. Remove the transition property from #output.
- D. FastAPI SSE requires WebSockets. Switch the endpoint to use WebSocket protocol.

**2. What bug does this code have?**

- A. EventSource does not support query parameters. The prompt must be sent in a POST body.
- B. The prompt is not URL-encoded. Special characters and spaces will break the query string.
- C. The API_BASE should be a relative path, not an absolute URL.
- D. Query parameters are not supported in HTTPS. This will only work in HTTP.

**3. Where does the fix go and what does it look like?**

- A. In the client TypeScript: add a 'mode: no-cors' option to every fetch() call.
- B. In the FastAPI backend: add CORSMiddleware with allow_origins including http://localhost:3000.
- C. In the browser: disable CORS checking in the browser security settings.
- D. In the HTML: add a <meta name='cors' content='allow'> tag.

**4. What is the correct approach?**

- A. Pass all parameters as query parameters in the EventSource URL.
- B. Use fetch() with response.body.getReader() to read the response body as a stream.
- C. Switch the endpoint to WebSocket, which supports both sending and receiving data.
- D. Base64-encode the JSON body and pass it as a single query parameter.

**5. What is causing this and how do you fix it?**

- A. EventSource auto-reconnects when the server closes the connection. Call source.close() when you receive the [DONE] sentinel.
- B. The server is sending too many events. Add a delay between tokens.
- C. The onmessage handler is being registered multiple times. Move it inside the EventSource constructor.
- D. EventSource should only be used with WebSocket upgrades. Switch to WebSocket.

**6. What is the correct fix that maintains strict type safety?**

- A. Change tsconfig to disable strictNullChecks.
- B. Cast with 'as HTMLDivElement' after getElementById: const el = document.getElementById('output') as HTMLDivElement;
- C. Use optional chaining: el?.textContent = 'hello';
- D. Add a null check: if (el !== null) { el.textContent = 'hello'; }

**7. What is the simplest way to serve both from FastAPI?**

- A. Run a separate nginx server on port 80 that proxies /api to FastAPI and serves /frontend statically.
- B. Add app.mount('/ui', StaticFiles(directory='frontend', html=True), name='ui') at the end of your FastAPI app.
- C. Copy index.html into the FastAPI templates directory and use Jinja2 to render it.
- D. Use subprocess to start python -m http.server from inside the FastAPI startup event.

**8. What is the correct response?**

- A. Agree immediately. React's component model always improves maintainability.
- B. Agree only if the team already knows React. Learning cost is the main factor.
- C. Evaluate the actual state complexity. 3 DOM elements is not enough complexity to justify a framework. Add React when you have 5+ components with shared state.
- D. Reject permanently. Frameworks should never be used in AI services.

_Answers are in `checks.json` in the lesson directory._